<a href="https://colab.research.google.com/github/martatolos/eae-dsaa/blob/main/nlp_advanced_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Engineering

> **Goal of the session:**
>
> - Learn practical techniques to write prompts that are reliable, precise, and easy to control.
> - Learn how to use prompting for different NLP tasks: classification, sentiment analysis, information extraction, and translation.
>
> **Scope of the session**
>
> - General prompting hints: writing clear instructions, include/exclude options, directive language, and structured outputs.
> - Extended techniques: few-shot prompting and chain-of-thought prompting.
> - We will use the OpenAI Responses API throughout.

## Setup

#### Dependencies

- `ipython`
- `openai` 2.40.0
- `pydantic`
- `python-dotenv`

In [ ]:
%pip install ipython openai==2.40.0 pydantic python-dotenv

### Imports

In [ ]:
import os

import dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel

### API Key

Add your OpenAI API key in the cell below or create a `.env` file in the same directory as this notebook with the following content:

```
OPENAI_API_KEY=your_openai_api_key
```

> [!Warning]
> Make sure you do not save or commit the file without removing your API key. If that happens, reset the key so that it is not compromised.

In [ ]:
open_ai_key = None
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", open_ai_key)

### Helper functions

In [ ]:
def get_completion(prompt: str, model_name: str = "gpt-4.1-nano") -> str:
    """Get the completion from OpenAI API.

    :param prompt: Prompt to be sent to the model.
    :param model_name: Name of the model to use. Defaults to "gpt-4.1-nano".
    :return: Completion text from the model.
    """
    return OpenAI().responses.create(model=model_name, input=prompt).output[0].content[0].text


def render_markdown(text: str) -> None:
    """Render text as markdown in the notebook."""
    display(Markdown(text))


def show_completion(prompt: str, model_name: str = "gpt-4.1-nano") -> None:
    """Get a completion and render it as markdown."""
    render_markdown(get_completion(prompt, model_name))


def get_structured_completion(prompt: str, schema: type[BaseModel], model_name: str = "gpt-4.1-nano") -> BaseModel:
    """Get a structured completion from OpenAI API.

    The SDK enforces the schema and returns a typed Pydantic object — no manual JSON parsing needed.

    :param prompt: Prompt to be sent to the model.
    :param schema: Pydantic model class defining the expected output structure.
    :param model_name: Name of the model to use. Defaults to "gpt-4.1-nano".
    :return: Parsed Pydantic model instance.
    """
    response = OpenAI().responses.parse(model=model_name, input=prompt, text_format=schema)
    return response.output_parsed

In [ ]:
prompt = "Tell me a joke"
show_completion(prompt)

---

## 1. General Prompting Hints

### 1.1 Writing Clear Instructions

Vague prompts give the model freedom to interpret your request in unexpected ways. The more specific you are about:

- **What** you want (content, format, length)
- **Who** the audience is
- **What constraints** apply

...the more predictable and useful the response will be. Clarity and specificity do not mean longer prompts — a concise but precise instruction beats a long, ambiguous one.

In [ ]:
# Vague prompt — the model decides the scope, format, and depth
prompt = "Tell me about marketing strategies"
show_completion(prompt)

In [ ]:
# Specific prompt — scope, audience, format, and length are all defined
prompt = """
List 3 digital marketing strategies suitable for a small B2B software company targeting European markets.
For each strategy:
- Provide a one-sentence description.
- Give one concrete first action the company could take.
- Indicate a realistic timeframe to see results.

Keep the total answer under 200 words and use bullet points.
"""
show_completion(prompt)

### 1.2 Include and Exclude Options

You can guide the model's output by explicitly telling it what to **include** and what to **leave out**.

- **Positive framing** (`"always include X"`, `"make sure to mention Y"`) steers the model towards specific content.
- **Negative framing** (`"do not mention Z"`, `"avoid discussing W"`) prevents unwanted content.

Using both together gives you fine-grained control over the output — especially useful when writing marketing copy, reports, or customer-facing text.

In [ ]:
# Without explicit include/exclude — the model makes its own choices
prompt = "Write a short product description for an AI-powered project management tool."
show_completion(prompt)

In [ ]:
# With explicit include and exclude — the output now covers exactly the right points
prompt = """
Write an 80-word product description for an AI-powered project management tool.

Include:
- Real-time collaboration across distributed teams
- Native integrations with Slack and Jira
- A mention of the 14-day free trial

Do NOT include:
- Any mention of pricing or subscription plans
- Names of competitor products
- Technical implementation details
"""
show_completion(prompt)

### 1.3 Forcing Behaviour with YOU MUST and CRITICAL

For constraints that are non-negotiable — legal requirements, brand guidelines, safety rules — emphatic directive language makes the model treat them as hard constraints:

- **`YOU MUST`** signals an absolute requirement that must always be fulfilled.
- **`CRITICAL`** flags a constraint that must never be violated.

This is especially useful in automated pipelines where you cannot manually review every output.

In [ ]:
complaint = """
I ordered a laptop three weeks ago and it still hasn't arrived. \
Your support team is completely useless and I want a full refund immediately!
"""

# Without explicit directives — the model may drift on key business constraints
prompt = f"""
You are a customer service agent. Reply to this customer complaint:

\"\"\"{complaint}\"\"\"
"""
show_completion(prompt)

In [ ]:
# With YOU MUST and CRITICAL — key constraints are locked in regardless of tone or phrasing
prompt = f"""
You are a customer service agent for an e-commerce company.

YOU MUST always maintain a calm and empathetic tone.
YOU MUST explicitly acknowledge the customer's frustration.
YOU MUST provide a concrete next step (e.g., a case reference number and a follow-up timeline).
CRITICAL: Never promise a specific delivery date.
CRITICAL: Never authorise a refund directly — always refer the customer to the returns team.

Reply to this customer complaint:

\"\"\"{complaint}\"\"\"
"""
show_completion(prompt)

### 1.4 Structured Outputs

So far, the model has returned free-form text. For applications that need to process the output programmatically — databases, dashboards, downstream pipelines — free-form text is difficult to work with reliably.

The OpenAI SDK supports **structured outputs**: you define a data schema using a [Pydantic](https://docs.pydantic.dev/) model, and the SDK guarantees that the response matches that schema exactly. No manual JSON parsing is needed.

The key method is `client.responses.parse()`, which accepts a `text_format` parameter containing your schema class. It returns a fully-typed Pydantic object.

In [ ]:
# Define the expected output schema as a Pydantic model
class Entities(BaseModel):
    persons: list[str]
    organisations: list[str]
    locations: list[str]

In [ ]:
text = """
During the annual health summit in Geneva, Sarah Chen, CEO of TechVision, announced a strategic
partnership with GlobalHealth Solutions. The agreement was co-signed by Dr. Marcus Weber
from the World Health Organization.
"""

prompt = f"Extract all persons, organisations, and locations from this text:\n\n{text}"

entities: Entities = get_structured_completion(prompt, Entities)

print("Persons:      ", entities.persons)
print("Organisations:", entities.organisations)
print("Locations:    ", entities.locations)

In [ ]:
# Fields are typed Python objects — iterate, index, or pass them to other functions directly
for person in entities.persons:
    print(f"Person found: {person}")

> **Note:** Unlike asking the model to "return JSON", structured outputs use schema validation at the API level. This makes your code robust against malformed or missing fields — the SDK retries automatically if the model produces output that does not conform to the schema.

### 1.5 Exercise

Work through the three tasks below. Each task has its own code cell.

**Task 1 — Entity extraction with a structured schema**

A press release snippet is provided below. Define a Pydantic model with fields `company: str`, `product: str`, and `announced_date: str`. Then write a prompt and call `get_structured_completion()` to extract those fields from the text.

In [ ]:
press_release = """
Barcelona, 12 May 2025 — NovaTech Solutions today unveiled its latest product,
DataPulse Pro, a real-time analytics platform designed for retail enterprises.
The company stated that DataPulse Pro will be available for early access starting June 2025.
"""

# Write your schema and prompt here

**Task 2 — Constrained generation**

Write a prompt that generates a 60-word hotel description. The description:
- YOU MUST mention the rooftop pool and the city-centre location.
- CRITICAL: must not mention price or competitor hotels.

Experiment with the constraints — try removing them one by one and observe how the output changes.

In [ ]:
# Write your prompt and code here

**Task 3 — Tone rewriting**

The angry customer review below needs to be rewritten as a polite, constructive message to the company's feedback team. Write a prompt that:
- Explicitly excludes aggressive or threatening language.
- Includes a concrete suggestion for improvement.

In [ ]:
angry_review = """
This app is an absolute disaster. It crashes every five minutes, loses my data,
and your support team never responds. I've wasted hours on this garbage.
I want my money back and I'll be telling everyone I know to avoid this product.
"""

# Write your prompt and code here

---

## 2. NLP Tasks and Applications

Large language models can solve a wide range of NLP tasks out of the box — no fine-tuning required.
The key is crafting a prompt that clearly describes the task.

In this section we explore four common NLP tasks:

- **Text classification** — assigning a category label to a piece of text.
- **Sentiment analysis** — determining the emotional tone of a text.
- **Information extraction** — pulling structured facts out of unstructured text.
- **Machine translation** — converting text from one language to another.

### 2.1 Text Classification

Text classification assigns a predefined category label to a piece of text.
Typical use cases include spam detection, topic labelling, and routing customer messages to the right team.

The prompt needs to:
1. Define the set of possible categories clearly.
2. Ask for only the label to keep the output easy to process.

In [ ]:
article = """
The European Central Bank raised interest rates by 25 basis points today,
citing persistent inflationary pressures across the eurozone. Economists
expect at least one further hike before the end of the year.
"""

categories = ["politics", "economics", "sports", "technology", "health"]

prompt = f"""
Classify the following news article into exactly one of these categories:
{", ".join(categories)}.

Respond with the category label only.

Article: \"\"\"{ article }\"\"\"
"""
show_completion(prompt)

### 2.2 Sentiment Analysis

Sentiment analysis is a specialised form of text classification that identifies the emotional tone of a text — typically positive, negative, or neutral. It is widely used in brand monitoring, product review analysis, and customer feedback pipelines.

Combined with structured outputs, we get a typed result we can use directly in code.

In [ ]:
class SentimentResult(BaseModel):
    sentiment: str   # "positive", "negative", or "neutral"
    confidence: float  # 0.0 to 1.0

reviews = [
    "The battery life is outstanding and the screen is gorgeous. Best phone I've owned.",
    "Arrived broken and customer support never replied to my emails. Complete waste of money.",
    "It's fine. Does what it says on the box, nothing more.",
]

for review in reviews:
    prompt = f"""
Analyse the sentiment of the following product review.
Return \"positive\", \"negative\", or \"neutral\" as the sentiment,
and a confidence score between 0.0 and 1.0.

Review: \"{review}\"
"""
    result = get_structured_completion(prompt, SentimentResult)
    print(f"Review:     {review[:65]}...")
    print(f"Sentiment:  {result.sentiment}  |  Confidence: {result.confidence:.2f}\n")

### 2.3 Information Extraction

Information extraction turns unstructured text into structured data by identifying specific facts — names, dates, amounts, relationships, and so on.

Combined with structured outputs (see Section 1.4), this becomes a reliable pipeline step: the Pydantic schema enforces exactly which fields are returned and in what format.

In [ ]:
class JobPosting(BaseModel):
    job_title: str
    company: str
    location: str
    required_skills: list[str]
    salary_range: str  # "not mentioned" if absent

posting = """
We are looking for a Senior Data Scientist to join our Madrid office at FinanceFlow Analytics.
The ideal candidate has strong Python skills, experience with machine learning frameworks such as
scikit-learn and PyTorch, and familiarity with SQL. Knowledge of financial modelling is a plus.
The role offers a competitive package between €65,000 and €85,000 per year.
"""

prompt = f"Extract the key information from this job posting:\n\n{posting}"

job = get_structured_completion(prompt, JobPosting)

print(f"Title:    {job.job_title}")
print(f"Company:  {job.company}")
print(f"Location: {job.location}")
print(f"Skills:   {', '.join(job.required_skills)}")
print(f"Salary:   {job.salary_range}")

### 2.4 Machine Translation

LLMs are trained on multilingual data and translate text across dozens of languages with high quality. Beyond literal translation, you can also control the formality or tone of the output.

In [ ]:
text = """
The quarterly earnings report shows a 12% increase in revenue compared to the same period
last year, driven primarily by strong performance in the software subscription segment.
"""

prompt = f"""
Translate the following English text into Spanish, French, and German.
Present the output as three clearly labelled sections.

Text: \"\"\"{text}\"\"\"\"
"""
show_completion(prompt)

### 2.5 Solving Multiple Tasks at Once

A single prompt can combine several NLP tasks, reducing the number of API calls and keeping context coherent. Define a Pydantic schema that captures all the outputs you need at once.

In [ ]:
class ReviewAnalysis(BaseModel):
    summary: str
    sentiment: str           # "positive", "negative", or "neutral"
    key_topics: list[str]
    suggested_response: str  # a short reply from the company

review = """
I've been using this accounting software for three months now. The invoicing module is
excellent and saves me hours each week. However, the mobile app is buggy and crashes
frequently. Also, the customer support chat took over two days to respond to my query.
I hope these issues get fixed because the core product is genuinely good.
"""

prompt = f"""
Analyse the following customer review and return:
1. A one-sentence summary.
2. The overall sentiment (positive, negative, or neutral).
3. The key topics mentioned (as a list).
4. A short, professional response the company could send to this customer.

Review: \"\"\"{review}\"\"\"\"
"""

analysis = get_structured_completion(prompt, ReviewAnalysis)

print(f"Summary:   {analysis.summary}")
print(f"Sentiment: {analysis.sentiment}")
print(f"Topics:    {', '.join(analysis.key_topics)}")
print(f"\nSuggested reply:\n{analysis.suggested_response}")

### 2.6 Exercise

Work through the three tasks below. Each task has its own code cell.

In [ ]:
# Task 1 — Multi-label topic classification
# The news snippet below may belong to more than one topic area.
# Define a Pydantic model with fields: primary_topic (str) and secondary_topics (list[str]).
# Write a prompt that extracts both fields using get_structured_completion().

news_snippet = """
The city council of Barcelona approved a €500 million investment plan yesterday to expand
the metro network and introduce electric bus fleets by 2028. The initiative is part of a broader
green transition strategy and is expected to create around 3,000 construction jobs.
"""

# Write your schema and prompt here

In [ ]:
# Task 2 — Multilingual sentiment analysis
# Three customer reviews in different languages are provided.
# Write a prompt that detects the language, translates the review to English,
# and classifies the sentiment — all in a single API call.
# Use a Pydantic model to capture: language (str), english_translation (str), sentiment (str).

reviews_multilingual = [
    "Das Produkt ist fantastisch! Sehr schnelle Lieferung und top Qualität.",
    "Très déçu par la qualité. Le produit est arrivé endomagé et le service client est inexistant.",
    "El precio es razonable pero la calidad podría mejorar bastante.",
]

# Write your schema and prompt here

In [ ]:
# Task 3 — Structured information extraction from a contract excerpt
# Extract the key terms from the clause below using a Pydantic model.
# Your model should capture: parties (list[str]), effective_date (str),
# payment_amount (str), and payment_schedule (str).

contract_excerpt = """
This Service Agreement is entered into as of 1 March 2025 by and between
Nexus Consulting Ltd ("Service Provider") and Orion Retail Group S.A. ("Client").
The Client agrees to pay the Service Provider a total fee of €24,000,
payable in equal monthly instalments of €2,000 on the first business day of each month,
commencing on 1 April 2025.
"""

# Write your schema and prompt here

---

## 3. Extended Prompting Techniques

### 3.1 Few-Shot Prompting

**Few-shot prompting** means including a small number of worked examples directly in your prompt. Instead of only describing what you want, you *show* the model with input-output pairs.

This is particularly effective for:
- Classification tasks where the label space needs to be defined precisely.
- Tasks where the desired output format is difficult to describe in words.
- Edge cases where the model's default behaviour is not quite right.

The more examples you provide, the more consistently the model follows the pattern — though there are diminishing returns beyond 4–6 examples for most tasks.

In [ ]:
ticket = "The export to CSV button does nothing when I click it. Tried on Chrome and Firefox."

# Zero-shot: no examples — the model guesses the categories from the description alone
prompt = f"""
Classify the following customer support ticket into exactly one of these categories:
bug, feature_request, compliment, other.

Respond with the category label only.

Ticket: \"\"\"{ticket}\"\"\"
"""
show_completion(prompt)

In [ ]:
# Two-shot: two examples anchor the model's understanding of the categories
prompt = f"""
Classify the following customer support ticket into exactly one of these categories:
bug, feature_request, compliment, other.

Respond with the category label only.

Example 1
Ticket: "The login page shows a blank screen after I enter my password."
Category: bug

Example 2
Ticket: "It would be great if the dashboard had a dark mode option."
Category: feature_request

Ticket: \"\"\"{ticket}\"\"\"
"""
show_completion(prompt)

In [ ]:
# Four-shot: all four categories are illustrated — better coverage of edge cases
prompt = f"""
Classify the following customer support ticket into exactly one of these categories:
bug, feature_request, compliment, other.

Respond with the category label only.

Example 1
Ticket: "The login page shows a blank screen after I enter my password."
Category: bug

Example 2
Ticket: "It would be great if the dashboard had a dark mode option."
Category: feature_request

Example 3
Ticket: "Just wanted to say the new onboarding flow is fantastic — super clear and fast!"
Category: compliment

Example 4
Ticket: "What are your office hours? I need to speak to someone on the phone."
Category: other

Ticket: \"\"\"{ticket}\"\"\"
"""
show_completion(prompt)

### 3.2 Chain-of-Thought Prompting

**Chain-of-thought (CoT) prompting** asks the model to reason step by step before giving a final answer. This works because it forces the model to allocate more computation to intermediate reasoning steps, reducing the chance of jumping to a wrong conclusion.

CoT is most impactful on:
- Multi-step arithmetic or financial calculations.
- Logical reasoning problems.
- Tasks where the correct answer depends on several intermediate conclusions.

We will demonstrate the effect using a multi-step business problem — and compare results across models and prompting styles.

In [ ]:
problem = """
A SaaS company has three pricing tiers:
- Basic: €10 per month
- Professional: €25 per month
- Enterprise: €60 per month

Current subscribers: 200 Basic, 80 Professional, 20 Enterprise.

After a targeted marketing campaign:
- 15% of Basic subscribers upgrade to Professional.
- 10% of Professional subscribers upgrade to Enterprise.

What is the total monthly revenue after the upgrades?
"""

# Older model, no chain-of-thought — prone to arithmetic errors on multi-step problems
prompt = f"""
Solve the following business problem. Give only the final answer.

{problem}
"""
render_markdown("**gpt-4o-mini (no CoT):**")
show_completion(prompt, model_name="gpt-4o-mini")

In [ ]:
# Latest model, same prompt — more capable models handle multi-step reasoning better even without CoT
render_markdown("**gpt-4.1 (no CoT):**")
show_completion(prompt, model_name="gpt-4.1")

In [ ]:
# Older model, with a simple chain-of-thought instruction — significant improvement
prompt_cot = f"""
Solve the following business problem. Think step by step before giving your final answer.

{problem}
"""
render_markdown("**gpt-4o-mini (with 'think step by step'):**")
show_completion(prompt_cot, model_name="gpt-4o-mini")

In [ ]:
# Structured CoT: explicit numbered steps guide the reasoning process — most reliable pattern
prompt_structured_cot = f"""
Solve the following business problem using this approach:

Step 1: Calculate how many Basic subscribers upgrade to Professional.
Step 2: Calculate how many Professional subscribers upgrade to Enterprise.
Step 3: Calculate the new number of subscribers in each tier.
Step 4: Calculate the monthly revenue for each tier.
Step 5: Sum the revenues to get the total monthly revenue.

Show your work for each step, then state the final answer clearly.

{problem}
"""
render_markdown("**gpt-4o-mini (structured CoT):**")
show_completion(prompt_structured_cot, model_name="gpt-4o-mini")

> **Takeaway:** The correct answer is **€5,930/month** (170 × €10 + 102 × €25 + 28 × €60). Notice how structured chain-of-thought prompting improves the accuracy of smaller, older models — sometimes to the level of a much larger one.

### 3.3 Exercise

Work through the three tasks below. Each task has its own code cell.

**Task 1 — Few-shot classification**

Three customer support tickets are provided below. Write a few-shot prompt to classify each into one of: `billing`, `technical`, `feedback`, `other`.

Try your prompt with **2 examples** and then with **4 examples**. Does the model handle edge cases more consistently with more examples?

In [ ]:
tickets = [
    "I was charged twice for my subscription this month. Please refund the duplicate charge.",
    "The mobile app freezes whenever I try to upload a file larger than 10 MB.",
    "Honestly the new interface is much cleaner — great improvement over the old design!",
]

# Write your prompt and code here

**Task 2 — Chain-of-thought for arithmetic**

Use the pricing problem below. First, ask the model for a direct answer (no CoT). Then rewrite the prompt with a step-by-step instruction. Compare the results — does CoT make a difference?

In [ ]:
exercise_problem = """
A retail company sells three product categories:
- Electronics: average margin of 12%, monthly sales of €80,000
- Clothing: average margin of 35%, monthly sales of €45,000
- Home & Garden: average margin of 28%, monthly sales of €30,000

Next month, a supplier discount increases the Electronics margin to 18%
and a seasonal promotion reduces the Clothing margin to 20%.

What is the total monthly gross profit before and after these changes,
and what is the difference in euros?
"""

# Write your prompts and code here (direct answer first, then with CoT)

**Task 3 — Chain-of-thought for logical reasoning**

Use the scheduling puzzle below. Ask for a direct answer first, then apply a structured chain-of-thought approach. Does prompting the model to reason step by step change the conclusion?

In [ ]:
puzzle = """
Four colleagues — Alice, Ben, Clara, and David — need to schedule a one-hour team meeting.
Their availability on Thursday is as follows:

- Alice is free from 9:00 to 12:00 and from 15:00 to 17:00.
- Ben is free from 10:00 to 13:00 and from 14:00 to 16:00.
- Clara is free from 9:00 to 11:00 and from 15:30 to 17:00.
- David is free from 10:30 to 12:30 and from 15:00 to 17:00.

What is the earliest one-hour slot on Thursday where all four colleagues are available?
"""

# Write your prompts and code here (direct answer first, then with CoT)